# Diabetes prediction — Mahdi's model v1

**BRFSS 2015 Diabetes Health Indicators**, class-balanced with SMOTE-ENN.

This notebook pulls two shared **W&B artifacts**, not local files, so all three of us
train and evaluate on byte-identical data no matter who ran `scripts/run_resample.py`:

- `brfss-smoteenn-resampled` — SMOTE-ENN'd training data. Train on this.
- `brfss-holdout-test` — untouched raw rows, split off *before* SMOTE-ENN ran. Evaluate
  headline metrics on this one; see the note in section 4 for why.

Working agreement: only edit notebooks inside `notebooks/mahdi/`. Shared code belongs
in `src/`, and changes there should be discussed first.

## 1. Pull the training and holdout datasets from the W&B artifacts

In [ ]:
# Run this first. It downloads the exact CSVs produced by scripts/run_resample.py:
# the resampled training set and the raw holdout set.
# Requires: `wandb login` (once per machine). See the README.
import sys
from pathlib import Path

# Repo root, so `src` and `configs` resolve regardless of where Jupyter started.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "configs").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import wandb
import yaml

OWNER = "mahdi"

cfg = yaml.safe_load((REPO_ROOT / "configs" / "resample_smoteenn.yaml").read_text())
TRAIN_ARTIFACT_NAME = cfg["artifact"]["name"]            # "brfss-smoteenn-resampled"
HOLDOUT_ARTIFACT_NAME = cfg["holdout_artifact"]["name"]  # "brfss-holdout-test"
TARGET = cfg["data"]["target_column"]                    # "diabetes_binary"

run = wandb.init(
    project=cfg["wandb"]["project"],
    entity=cfg["wandb"]["entity"],
    job_type="train",
    name=f"{OWNER}-train-v1",
    tags=[OWNER, "modelling"],
)

# ":latest" always resolves to the newest version of the artifact. Pin an explicit
# version (e.g. ":v0") once you want frozen results -- pin BOTH artifacts together,
# since they're produced by the same resampling run.
train_artifact = run.use_artifact(f"{TRAIN_ARTIFACT_NAME}:latest")
train_dir = Path(train_artifact.download())
train_df = pd.read_csv(next(train_dir.glob("*.csv")))

holdout_artifact = run.use_artifact(f"{HOLDOUT_ARTIFACT_NAME}:latest")
holdout_dir = Path(holdout_artifact.download())
holdout_df = pd.read_csv(next(holdout_dir.glob("*.csv")))

print(f"train   : {TRAIN_ARTIFACT_NAME}:{train_artifact.version}   shape={train_df.shape}")
print(f"holdout : {HOLDOUT_ARTIFACT_NAME}:{holdout_artifact.version}   shape={holdout_df.shape}")
train_df.head()

## 2. Sanity check — confirm which set is which

In [ ]:
train_dist = train_df[TARGET].value_counts().sort_index()
print("train (post-SMOTE-ENN):")
print(train_dist.to_string())
print(f"positive rate: {train_df[TARGET].mean():.4f}  (well above raw ~0.139 -- it's resampled)\n")

holdout_dist = holdout_df[TARGET].value_counts().sort_index()
print("holdout (raw, untouched):")
print(holdout_dist.to_string())
print(f"positive rate: {holdout_df[TARGET].mean():.4f}  (should be close to raw BRFSS's ~0.139)")
print("(if these two look swapped, you loaded the wrong artifact into the wrong variable)")

## 3. Build features and target

No `train_test_split` here — the split already happened, on the raw data, *before*
SMOTE-ENN ran (inside `scripts/run_resample.py`). `train_df` and `holdout_df` are
already the two halves; splitting again here would defeat the point.

In [ ]:
from src.utils.seed import set_seed

SEED = cfg["random_seed"]
set_seed(SEED)

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET].astype(int)

X_test = holdout_df.drop(columns=[TARGET])
y_test = holdout_df[TARGET].astype(int)

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")

## 4. Mahdi's modelling — start here

Nothing below is implemented yet; this is the handoff point.

A couple of things worth keeping in mind as you experiment:

- **`X_test`/`y_test` are real, untouched rows** — pulled from `brfss-holdout-test`,
  split off before SMOTE-ENN ever ran. Report headline metrics on this set; it's the
  shared evaluation protocol for all three of us, so results are directly comparable.
  (`X_train`/`y_train` are the SMOTE-ENN'd rows — fine to train on, don't evaluate on
  them.)
- Log metrics with `run.log({...})` so all three runs land in the same W&B project and
  can be compared side by side.
- Call `run.finish()` when you're done with the notebook.

### 4.1 Setup — imports, palette, model factory, metrics helper

Everything below only *fits* on `X_train`/`y_train`. `X_test`/`y_test` (the raw
holdout) are used exclusively via `.predict()`/`.predict_proba()` — never
`.fit()`, `.fit_transform()`, or any feature-selection call.

In [ ]:
# Imports for this section. `pandas`/`numpy`/`wandb` are already available from
# earlier cells; everything else needed for modelling + plotting goes here.
%matplotlib inline
# 'retina' renders inline figures at 2x pixel density -- without this, charts look
# pixelated/blurry on any HiDPI display (the default inline backend does not do this).
%config InlineBackend.figure_format = 'retina'
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, balanced_accuracy_score, confusion_matrix,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Fixed categorical colors, one hue per model, assigned once and never reused for
# anything else -- and a muted neutral for "not highlighted" bars.
MODEL_COLORS = {
    "logreg": "#2a78d6",         # blue
    "random_forest": "#eb6834",  # orange
    "xgboost": "#1baf7a",        # aqua
    "lightgbm": "#e87ba4",       # magenta
    "knn": "#eda100",            # yellow
}
MODEL_ORDER = list(MODEL_COLORS)
MUTED = "#c7c6c0"
# Diverging blue<->red through a neutral gray midpoint, for the correlation heatmap
# (correlation is signed and centered at 0 -- a single-hue or rainbow map would
# misrepresent that).
DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "diverging_blue_red", ["#2a78d6", "#f0efec", "#e34948"]
)
# Sequential single-hue (light -> dark blue), for the confusion-matrix grid --
# those cells encode a magnitude (count), not a sign.
SEQUENTIAL_CMAP = LinearSegmentedColormap.from_list(
    "sequential_blue", ["#cde2fb", "#2a78d6", "#0d366b"]
)

# Plausibly *downstream* of having diabetes rather than causal risk factors --
# used later for the sensitivity pass.
PROXY_COLS = ["genhlth", "menthlth", "physhlth", "diffwalk"]


def make_models(seed):
    """Fresh, unfitted model instances -- called once per feature group so no
    model or fitted scaler ever carries state across groups."""
    return {
        "logreg": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=seed, n_jobs=-1)),
        ]),
        "random_forest": RandomForestClassifier(random_state=seed, n_jobs=-1),
        "xgboost": XGBClassifier(random_state=seed, n_jobs=-1, eval_metric="logloss"),
        "lightgbm": LGBMClassifier(random_state=seed, n_jobs=-1, verbose=-1),
        "knn": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_jobs=-1)),
        ]),
    }


def compute_metrics(y_true, y_pred, y_proba):
    return {
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def log_and_show(fig, key, filename):
    """Save a compact figure, log it to W&B, display it inline, and close it."""
    fig.savefig(filename, dpi=220, bbox_inches="tight")
    run.log({key: wandb.Image(filename)})
    plt.show()
    plt.close(fig)


print("Models:", MODEL_ORDER)
print("Proxy columns (excluded in the sensitivity pass):", PROXY_COLS)

### 4.2 Compact EDA — sanity checks, not a report

In [ ]:
# Class balance: train (resampled) vs. test (original), side by side.
fig, ax = plt.subplots(figsize=(5, 3))
rates = [y_train.mean(), y_test.mean()]
labels = ["train\n(SMOTE-ENN)", "test\n(raw holdout)"]
colors = [MODEL_COLORS["logreg"], MODEL_COLORS["random_forest"]]
bars = ax.bar(labels, rates, color=colors, width=0.5)
for b, r in zip(bars, rates):
    ax.text(b.get_x() + b.get_width() / 2, r + 0.02, f"{r:.3f}", ha="center", fontsize=8)
ax.set_ylim(0, 1)
ax.set_ylabel("positive rate")
ax.set_title("Class balance: train vs. test", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
log_and_show(fig, "eda/class_balance", "eda_class_balance.png")

In [ ]:
# Correlation within the biological group only, to check for redundant features
# before trusting per-feature importances later. Diverging colormap, no annot --
# this is a sanity check, not a figure for a report.
bio_cols = ["bmi", "highchol", "cholcheck", "highbp", "heartdiseaseorattack", "stroke", "age", "sex"]
corr = X_train[bio_cols].corr()

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(corr, cmap=DIVERGING_CMAP, vmin=-1, vmax=1)
ax.set_xticks(range(len(bio_cols)))
ax.set_xticklabels(bio_cols, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(len(bio_cols)))
ax.set_yticklabels(bio_cols, fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8, label="corr")
ax.set_title("Biological group — feature correlation", fontsize=10)
fig.tight_layout()
log_and_show(fig, "eda/biological_corr", "eda_biological_corr.png")

### 4.3 Feature groups & mutual information

Domain groups, plus a hybrid top-10 selected two independent ways — mutual
information and RFE. Both are fit on `X_train` only.

In [ ]:
def build_domain_groups(exclude=()):
    """Domain feature groups, optionally dropping columns (used by the
    sensitivity pass). `combined_all` is every remaining column in X_train's
    own order -- derived, not a hardcoded union, so it can't drift out of sync."""
    exclude = set(exclude)
    groups = {
        "biological": [c for c in
            ["bmi", "highchol", "cholcheck", "highbp", "heartdiseaseorattack", "stroke", "age", "sex"]
            if c not in exclude],
        "socioeconomic": [c for c in
            ["income", "education", "anyhealthcare", "nodocbccost"]
            if c not in exclude],
        "lifestyle": [c for c in
            ["smoker", "physactivity", "fruits", "veggies", "hvyalcoholconsump",
             "menthlth", "physhlth", "diffwalk", "genhlth"]
            if c not in exclude],
    }
    groups["combined_all"] = [c for c in X_train.columns if c not in exclude]
    return groups


def compute_hybrid_top10(pool_cols, seed):
    """Top-10 features from `pool_cols` two ways, fit on X_train only:
    mutual_info_classif ranking, and RFE with a scaled LogisticRegression.
    Returns (mi_series_full, top10_mi, top10_rfe)."""
    X_pool = X_train[pool_cols]

    mi_scores = mutual_info_classif(X_pool, y_train, random_state=seed)
    mi_series = pd.Series(mi_scores, index=pool_cols).sort_values(ascending=False)
    top10_mi = list(mi_series.index[:10])

    rfe_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("rfe", RFE(LogisticRegression(max_iter=1000, random_state=seed), n_features_to_select=10)),
    ])
    rfe_pipe.fit(X_pool, y_train)
    support = rfe_pipe.named_steps["rfe"].support_
    top10_rfe = [str(c) for c in np.array(pool_cols)[support]]  # str(): drop numpy's str_ wrapper

    return mi_series, top10_mi, top10_rfe

In [ ]:
mi_series_full, top10_mi, top10_rfe = compute_hybrid_top10(list(X_train.columns), SEED)

print(f"{'MI top10':<12}: {top10_mi}")
print(f"{'RFE top10':<12}: {top10_rfe}")
print(f"{'overlap':<12}: {sorted(set(top10_mi) & set(top10_rfe))}")

# Horizontal bar, sorted descending (top of chart = highest MI), top 10 highlighted.
mi_asc = mi_series_full.sort_values(ascending=True)
bar_colors = [MODEL_COLORS["logreg"] if f in top10_mi else MUTED for f in mi_asc.index]

fig, ax = plt.subplots(figsize=(5, 4))
ax.barh(mi_asc.index, mi_asc.values, color=bar_colors)
ax.set_xlabel("mutual information")
ax.set_title("Mutual information — all features (top 10 highlighted)", fontsize=10)
ax.tick_params(axis="y", labelsize=7)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
log_and_show(fig, "feature_selection/mi_scores", "mi_scores.png")

run.log({
    "feature_selection/mi_table": wandb.Table(
        dataframe=mi_series_full.rename_axis("feature").reset_index(name="mi_score")
    )
})

domain_groups = build_domain_groups()
domain_groups["hybrid_mi"] = top10_mi
domain_groups["hybrid_rfe"] = top10_rfe
GROUP_ORDER = list(domain_groups)
print("\nFeature groups:", {k: len(v) for k, v in domain_groups.items()})

### 4.4 Run every model on every feature group (full feature set)

In [ ]:
def run_experiment(groups, seed):
    """Fit each model on each group's train columns, evaluate on the matching
    holdout columns. X_test is only ever indexed and predicted on here.
    Returns (results_df, confusion_matrices) -- cms keyed by (group, model),
    plotted as a compact grid rather than printed as raw arrays."""
    rows, cms = [], {}
    for group_name in groups:
        cols = groups[group_name]
        X_tr, X_te = X_train[cols], X_test[cols]
        models = make_models(seed)
        for model_name in MODEL_ORDER:
            model = models[model_name]
            model.fit(X_tr, y_train)
            y_pred = model.predict(X_te)
            y_proba = model.predict_proba(X_te)[:, 1]

            cms[(group_name, model_name)] = confusion_matrix(y_test, y_pred)
            rows.append({
                "group": group_name,
                "model": model_name,
                **compute_metrics(y_test, y_pred, y_proba),
            })
    return pd.DataFrame(rows), cms


def plot_confusion_grid(cms, groups_order, models_order, title):
    """One compact figure: a grid of small confusion-matrix heatmaps, rows=groups,
    columns=models -- instead of printing len(groups)*len(models) raw arrays."""
    n_rows, n_cols = len(groups_order), len(models_order)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.5 * n_cols, 1.5 * n_rows), squeeze=False)
    for i, group_name in enumerate(groups_order):
        for j, model_name in enumerate(models_order):
            ax = axes[i, j]
            cm = cms[(group_name, model_name)]
            ax.imshow(cm, cmap=SEQUENTIAL_CMAP, vmin=0)
            for (r, c), val in np.ndenumerate(cm):
                ax.text(c, r, str(val), ha="center", va="center", fontsize=6,
                        color="white" if val > cm.max() / 2 else "#0b0b0b")
            ax.set_xticks([])
            ax.set_yticks([])
            if i == 0:
                ax.set_title(model_name, fontsize=7)
            if j == 0:
                ax.set_ylabel(group_name, fontsize=7, rotation=0, ha="right", va="center")
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    return fig

In [ ]:
results_df, cms_main = run_experiment(domain_groups, SEED)
results_df.to_csv("results_group_models.csv", index=False)

print("\nResults, sorted by PR-AUC (the metric that matters most -- test is imbalanced):")
print(results_df.sort_values("pr_auc", ascending=False).to_string(index=False))

run.log({"results/table": wandb.Table(dataframe=results_df)})

results_artifact = wandb.Artifact(
    "group-model-results",
    type="results",
    description=(
        "Precision/recall/F1/ROC-AUC/PR-AUC/balanced-accuracy per feature-group x "
        "model, evaluated on the raw holdout set (brfss-holdout-test). Lets Sadman "
        "and Abrar see the comparison without re-running the notebook."
    ),
)
results_artifact.add_file("results_group_models.csv")
run.log_artifact(results_artifact)

fig_cm = plot_confusion_grid(cms_main, list(domain_groups), MODEL_ORDER, "Confusion matrices — full feature set")
log_and_show(fig_cm, "results/confusion_grid", "results_confusion_grid.png")

In [ ]:
def plot_grouped_bar(df, metric, title):
    groups_order = [g for g in GROUP_ORDER if g in df["group"].unique()]
    x = np.arange(len(groups_order))
    n_models = len(MODEL_ORDER)
    width = 0.8 / n_models

    fig, ax = plt.subplots(figsize=(7, 4))
    for i, model_name in enumerate(MODEL_ORDER):
        vals = [
            df.loc[(df["group"] == g) & (df["model"] == model_name), metric].iloc[0]
            for g in groups_order
        ]
        ax.bar(x + i * width - 0.4 + width / 2, vals, width=width,
               label=model_name, color=MODEL_COLORS[model_name])

    ax.set_xticks(x)
    ax.set_xticklabels(groups_order, rotation=20, ha="right", fontsize=8)
    ax.set_ylabel(metric)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7, frameon=False, ncol=3)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    return fig


fig_roc = plot_grouped_bar(results_df, "roc_auc", "ROC-AUC by feature group and model")
log_and_show(fig_roc, "results/roc_auc_chart", "results_roc_auc.png")

fig_pr = plot_grouped_bar(results_df, "pr_auc", "PR-AUC by feature group and model")
log_and_show(fig_pr, "results/pr_auc_chart", "results_pr_auc.png")

### 4.5 Sensitivity pass — drop the possible diabetes-downstream proxies

`genhlth`, `menthlth`, `physhlth`, `diffwalk` are plausibly *downstream* of
already having diabetes rather than independent risk factors, so they could be
inflating scores relative to a model meant to flag risk before diagnosis.
Rerun everything — domain groups, MI, RFE, all four models — with those four
columns removed from the candidate pool entirely, not just filtered out of the
previous results.

In [ ]:
pool_no_proxy = [c for c in X_train.columns if c not in PROXY_COLS]
mi_series_np, top10_mi_np, top10_rfe_np = compute_hybrid_top10(pool_no_proxy, SEED)

print(f"{'MI top10 (no-proxy)':<22}: {top10_mi_np}")
print(f"{'RFE top10 (no-proxy)':<22}: {top10_rfe_np}")

no_proxy_groups = build_domain_groups(exclude=PROXY_COLS)
no_proxy_groups["hybrid_mi"] = top10_mi_np
no_proxy_groups["hybrid_rfe"] = top10_rfe_np
print("\nFeature groups (no-proxy):", {k: len(v) for k, v in no_proxy_groups.items()})

results_no_proxy_df, cms_no_proxy = run_experiment(no_proxy_groups, SEED)
results_no_proxy_df.to_csv("results_group_models_no_proxy.csv", index=False)

print("\nResults (no-proxy), sorted by PR-AUC:")
print(results_no_proxy_df.sort_values("pr_auc", ascending=False).to_string(index=False))

run.log({"results_no_proxy/table": wandb.Table(dataframe=results_no_proxy_df)})

no_proxy_artifact = wandb.Artifact(
    "group-model-results-no-proxy",
    type="results",
    description=(
        "Same experiment as group-model-results, with genhlth/menthlth/physhlth/"
        "diffwalk removed from every group as a sensitivity check for "
        "diabetes-downstream proxy features."
    ),
)
no_proxy_artifact.add_file("results_group_models_no_proxy.csv")
run.log_artifact(no_proxy_artifact)

fig_cm_np = plot_confusion_grid(
    cms_no_proxy, list(no_proxy_groups), MODEL_ORDER, "Confusion matrices — no proxy features"
)
log_and_show(fig_cm_np, "results_no_proxy/confusion_grid", "results_confusion_grid_no_proxy.png")

fig_roc_np = plot_grouped_bar(
    results_no_proxy_df, "roc_auc", "ROC-AUC by feature group and model (no proxy features)"
)
log_and_show(fig_roc_np, "results_no_proxy/roc_auc_chart", "results_roc_auc_no_proxy.png")

fig_pr_np = plot_grouped_bar(
    results_no_proxy_df, "pr_auc", "PR-AUC by feature group and model (no proxy features)"
)
log_and_show(fig_pr_np, "results_no_proxy/pr_auc_chart", "results_pr_auc_no_proxy.png")

### 4.6 Did the feature-group ranking change?

In [ ]:
def group_ranking(df):
    return df.groupby("group")["pr_auc"].mean().reindex(GROUP_ORDER).sort_values(ascending=False)


rank_main = group_ranking(results_df)
rank_no_proxy = group_ranking(results_no_proxy_df)

print("Group ranking by mean PR-AUC (with proxy features):   ", list(rank_main.index))
print("Group ranking by mean PR-AUC (without proxy features):", list(rank_no_proxy.index))

changed = list(rank_main.index) != list(rank_no_proxy.index)
print(f"\nGroup ranking {'CHANGED' if changed else 'did NOT change'} after removing the proxy features.")

In [ ]:
run.finish()